In [1]:
import sys
print(sys.executable)

/Users/shashankjaji/Documents/Development/My Portfolio Projects /ppg-hr-estimation/.venv/bin/python


In [2]:
from pathlib import Path

RAW = Path("../data/raw/PPG_FieldStudy")
print(RAW.exists())
print(sorted(p.name for p in RAW.iterdir())[:5])

True
['.DS_Store', 'PPG_FieldStudy_readme.pdf', 'S1', 'S10', 'S11']


In [3]:
import pickle

def load_subject(subject_id: int):
    path = RAW / f"S{subject_id}" / f"S{subject_id}.pkl"
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")

s1 = load_subject(1)
print(type(s1))
print(s1.keys())

<class 'dict'>
dict_keys(['rpeaks', 'signal', 'label', 'activity', 'questionnaire', 'subject'])


In [4]:
import numpy as np

def describe(obj, indent=0):
    pad = "  " * indent
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(f"{pad}{k}:")
            describe(v, indent + 1)
    elif isinstance(obj, np.ndarray):
        print(f"{pad}ndarray shape={obj.shape} dtype={obj.dtype}")
    else:
        print(f"{pad}{type(obj).__name__}: {str(obj)[:60]}")

describe(s1)

rpeaks:
  ndarray shape=(11431,) dtype=int32
signal:
  chest:
    ACC:
      ndarray shape=(6448400, 3) dtype=float64
    ECG:
      ndarray shape=(6448400, 1) dtype=float64
    EMG:
      ndarray shape=(6448400, 1) dtype=float64
    EDA:
      ndarray shape=(6448400, 1) dtype=float64
    Temp:
      ndarray shape=(6448400, 1) dtype=float32
    Resp:
      ndarray shape=(6448400, 1) dtype=float64
  wrist:
    ACC:
      ndarray shape=(294784, 3) dtype=float64
    BVP:
      ndarray shape=(589568, 1) dtype=float64
    EDA:
      ndarray shape=(36848, 1) dtype=float64
    TEMP:
      ndarray shape=(36848, 1) dtype=float64
label:
  ndarray shape=(4603,) dtype=float64
activity:
  ndarray shape=(36848, 1) dtype=float64
questionnaire:
  WEIGHT:
    float: 78.0
  Gender:
    str:  m
  AGE:
    int: 34
  HEIGHT:
    float: 182.0
  SKIN:
    int: 3
  SPORT:
    int: 6
subject:
  str: S1


In [5]:
bvp = s1["signal"]["wrist"]["BVP"][:, 0]
acc = s1["signal"]["wrist"]["ACC"]
labels = s1["label"]
activity = s1["activity"][:, 0]

FS_BVP, FS_ACC, FS_ACT = 64, 32, 4

dur = len(bvp) / FS_BVP
n_expected = int((dur - 8) / 2) + 1

print(f"duration:        {dur:.1f} s ({dur/60:.1f} min)")
print(f"acc duration:    {len(acc)/FS_ACC:.1f} s")
print(f"activity dur:    {len(activity)/FS_ACT:.1f} s")
print(f"labels: {len(labels)}  expected: {n_expected}")
print(f"HR range: {labels.min():.1f} – {labels.max():.1f} BPM")
print(f"HR mean:  {labels.mean():.1f} BPM")

duration:        9212.0 s (153.5 min)
acc duration:    9212.0 s
activity dur:    9212.0 s
labels: 4603  expected: 4603
HR range: 41.9 – 150.2 BPM
HR mean:  74.8 BPM


In [6]:
def window_slice(i, signal, fs, win_sec=8, shift_sec=2):
    """Return the signal segment corresponding to label index i."""
    start = int(i * shift_sec * fs)
    stop = start + int(win_sec * fs)
    return signal[start:stop]

i = 1000
print(f"label[{i}] = {labels[i]:.1f} BPM")
print(f"covers seconds {i*2} – {i*2+8}")
print(f"BVP segment: {window_slice(i, bvp, FS_BVP).shape}  (expect 512)")
print(f"ACC segment: {window_slice(i, acc, FS_ACC).shape}  (expect (256, 3))")

label[1000] = 102.3 BPM
covers seconds 2000 – 2008
BVP segment: (512,)  (expect 512)
ACC segment: (256, 3)  (expect (256, 3))


In [1]:
from ppghr.io import load_subject, wrist_streams, window_slice, n_windows, FS_BVP, ACTIVITIES

s1 = load_subject(1)
bvp, acc, labels, activity = wrist_streams(s1)
print(len(labels), n_windows(bvp))
print(window_slice(1000, bvp, FS_BVP).shape)

4603 4603
(512,)
